In [0]:
USE CATALOG workspace;
USE SCHEMA causal_project;

## Silver Table Overview
The **silver table** is a cleaned and enriched dataset, combining customer transactions, campaign information, demographics, product attributes, and promotional features into a single analytics-ready table. The goal is to create a customer-level view that can be used for causal inference, campaign effectiveness analysis, and machine learning.

### Joins Performed
The silver table is created by joining several bronze-layer datasets:

1. transactions → clean campaign table (household_key)

Associates purchases with campaign exposure periods and treatment labels

2. transactions → demographics (household_key)

Adds household demographic characteristics

3. transactions → products (product_id)

Enriches transactions with product hierarchy, brand, manufacturer, and category information

4. transactions → features (store_id, week_no, product_id)

Adds promotional activity such as mailers and in-store displays

5. campaigns → campaign descriptions (campaign)

Retrieves campaign start/end dates and campaign types

In [0]:
-- 1. Row count and distinct households, expecting 760 distinct households
SELECT 
  COUNT(*)                        AS total_rows,
  COUNT(DISTINCT household_key)   AS distinct_households
FROM (
  WITH ordered_campaigns AS (
  SELECT 
    b.household_key, 
    b.campaign, 
    b.description, 
    d.start_day,
    d.end_day,
    ROW_NUMBER() OVER (
      PARTITION BY b.household_key 
      ORDER BY d.start_day
    ) AS rn 
  FROM bronze_campaigns b 
  JOIN bronze_campaign_desc d ON b.campaign = d.campaign
  WHERE household_key IN (
    SELECT household_key FROM bronze_demographics
  )
),

first_only AS (
  SELECT * FROM ordered_campaigns WHERE rn = 1
),

next_campaign AS (
    SELECT f.household_key, MIN(other_d.start_day) - 1 AS clean_end_day
    FROM first_only f   
    JOIN bronze_campaigns other ON other.household_key = f.household_key
    AND other.campaign    != f.campaign JOIN bronze_campaign_desc other_d
    ON other_d.campaign    = other.campaign
    AND other_d.start_day <= f.end_day
    AND other_d.end_day   >= f.start_day GROUP BY f.household_key),

clean AS (
      SELECT
    f.household_key,
    f.campaign,
    f.description,
    f.start_day,
    f.end_day                                             AS original_end_day,
    COALESCE(n.clean_end_day, f.end_day)                  AS clean_end_day,
    CASE WHEN n.clean_end_day IS NOT NULL 
         THEN 1 ELSE 0 END                               AS was_truncated,
    COALESCE(n.clean_end_day, f.end_day) 
      - f.start_day + 1                                  AS clean_window_days,
    CASE WHEN f.description = 'TypeA' 
         THEN 1 ELSE 0 END                               AS treatment
  FROM first_only f
  LEFT JOIN next_campaign n ON f.household_key = n.household_key
)

SELECT 
  t.*, p.department, p.commodity_desc, p.sub_commodity_desc, 
p.manufacturer, p.brand, p.curr_size_of_product, ft.mailer,ft.display,dg.classification_1,
dg.classification_2, dg.classification_3,dg.HOMEOWNER_DESC,dg.classification_5,dg.classification_4,dg.KID_CATEGORY_DESC,
  -- Treatment
  c.treatment,
  c.description                                           AS campaign_type,
  c.clean_window_days,
  c.was_truncated,
  c.start_day                                             AS campaign_start,
  c.clean_end_day                                         AS campaign_end,

  -- Window flags
  CASE WHEN t.day BETWEEN c.start_day AND c.clean_end_day 
       THEN 1 ELSE 0 END                                  AS in_campaign_window,
  CASE WHEN t.day < c.start_day 
       THEN 1 ELSE 0 END                                  AS pre_campaign,

  -- Customer spend (actual price paid, not retailer revenue)
  CASE WHEN t.sales_value < ABS(t.coupon_disc) THEN 0 ELSE t.sales_value + t.coupon_disc END     AS customer_spend,

  -- Coupon usage flag
  CASE WHEN t.coupon_disc != 0 
       THEN 1 ELSE 0 END                                  AS coupon_used,
    
    CASE WHEN retail_disc != 0 
     THEN 1 ELSE 0 END                           AS loyalty_card_used

FROM bronze_transactions t 
JOIN clean c ON t.household_key = c.household_key
JOIN bronze_demographics dg on t.household_key = dg.household_key
JOIN bronze_products p on t.PRODUCT_ID = p.PRODUCT_ID
LEFT JOIN bronze_features ft on t.STORE_ID = ft.STORE_ID AND t.WEEK_NO = ft.WEEK_NO AND t.PRODUCT_ID = ft.PRODUCT_ID
)

In [0]:
 -- 2. Treatment distribution, expecting TypeA=1, TypeB=0, TypeC=0: ~512 TypeA households, ~248 TypeB/TypeC households
SELECT 
    treatment, campaign_type, COUNT(DISTINCT household_key) as households, COUNT(*) AS transaction_rows
FROM (
  WITH ordered_campaigns AS (
  SELECT 
    b.household_key, 
    b.campaign, 
    b.description, 
    d.start_day,
    d.end_day,
    ROW_NUMBER() OVER (
      PARTITION BY b.household_key 
      ORDER BY d.start_day
    ) AS rn 
  FROM bronze_campaigns b 
  JOIN bronze_campaign_desc d ON b.campaign = d.campaign
  WHERE household_key IN (
    SELECT household_key FROM bronze_demographics
  )
),

first_only AS (
  SELECT * FROM ordered_campaigns WHERE rn = 1
),

next_campaign AS (
    SELECT f.household_key, MIN(other_d.start_day) - 1 AS clean_end_day
    FROM first_only f   
    JOIN bronze_campaigns other ON other.household_key = f.household_key
    AND other.campaign    != f.campaign JOIN bronze_campaign_desc other_d
    ON other_d.campaign    = other.campaign
    AND other_d.start_day <= f.end_day
    AND other_d.end_day   >= f.start_day GROUP BY f.household_key),

clean AS (
      SELECT
    f.household_key,
    f.campaign,
    f.description,
    f.start_day,
    f.end_day                                             AS original_end_day,
    COALESCE(n.clean_end_day, f.end_day)                  AS clean_end_day,
    CASE WHEN n.clean_end_day IS NOT NULL 
         THEN 1 ELSE 0 END                               AS was_truncated,
    COALESCE(n.clean_end_day, f.end_day) 
      - f.start_day + 1                                  AS clean_window_days,
    CASE WHEN f.description = 'TypeA' 
         THEN 1 ELSE 0 END                               AS treatment
  FROM first_only f
  LEFT JOIN next_campaign n ON f.household_key = n.household_key
)

SELECT 
  t.*, p.department, p.commodity_desc, p.sub_commodity_desc, 
p.manufacturer, p.brand, p.curr_size_of_product, ft.mailer,ft.display,dg.classification_1,
dg.classification_2, dg.classification_3,dg.HOMEOWNER_DESC,dg.classification_5,dg.classification_4,dg.KID_CATEGORY_DESC,
  -- Treatment
  c.treatment,
  c.description                                           AS campaign_type,
  c.clean_window_days,
  c.was_truncated,
  c.start_day                                             AS campaign_start,
  c.clean_end_day                                         AS campaign_end,

  -- Window flags
  CASE WHEN t.day BETWEEN c.start_day AND c.clean_end_day 
       THEN 1 ELSE 0 END                                  AS in_campaign_window,
  CASE WHEN t.day < c.start_day 
       THEN 1 ELSE 0 END                                  AS pre_campaign,

  -- Customer spend (actual price paid, not retailer revenue)
  t.sales_value + t.coupon_disc      AS customer_spend,

  -- Coupon usage flag
  CASE WHEN t.coupon_disc != 0 
       THEN 1 ELSE 0 END                                  AS coupon_used,
    
    CASE WHEN retail_disc != 0 
     THEN 1 ELSE 0 END                           AS loyalty_card_used

FROM bronze_transactions t 
JOIN clean c ON t.household_key = c.household_key
JOIN bronze_demographics dg on t.household_key = dg.household_key
JOIN bronze_products p on t.PRODUCT_ID = p.PRODUCT_ID
LEFT JOIN bronze_features ft on t.STORE_ID = ft.STORE_ID AND t.WEEK_NO = ft.WEEK_NO AND t.PRODUCT_ID = ft.PRODUCT_ID
)
GROUP BY treatment, campaign_type

In [0]:
 -- 3. Window flags never conflict, should never see in_campaign_window=1 AND pre_campaign=1 simultaneously
SELECT 
    in_campaign_window, pre_campaign, COUNT(*) AS transaction_rows
FROM (
  WITH ordered_campaigns AS (
  SELECT 
    b.household_key, 
    b.campaign, 
    b.description, 
    d.start_day,
    d.end_day,
    ROW_NUMBER() OVER (
      PARTITION BY b.household_key 
      ORDER BY d.start_day
    ) AS rn 
  FROM bronze_campaigns b 
  JOIN bronze_campaign_desc d ON b.campaign = d.campaign
  WHERE household_key IN (
    SELECT household_key FROM bronze_demographics
  )
),

first_only AS (
  SELECT * FROM ordered_campaigns WHERE rn = 1
),

next_campaign AS (
    SELECT f.household_key, MIN(other_d.start_day) - 1 AS clean_end_day
    FROM first_only f   
    JOIN bronze_campaigns other ON other.household_key = f.household_key
    AND other.campaign    != f.campaign JOIN bronze_campaign_desc other_d
    ON other_d.campaign    = other.campaign
    AND other_d.start_day <= f.end_day
    AND other_d.end_day   >= f.start_day GROUP BY f.household_key),

clean AS (
      SELECT
    f.household_key,
    f.campaign,
    f.description,
    f.start_day,
    f.end_day                                             AS original_end_day,
    COALESCE(n.clean_end_day, f.end_day)                  AS clean_end_day,
    CASE WHEN n.clean_end_day IS NOT NULL 
         THEN 1 ELSE 0 END                               AS was_truncated,
    COALESCE(n.clean_end_day, f.end_day) 
      - f.start_day + 1                                  AS clean_window_days,
    CASE WHEN f.description = 'TypeA' 
         THEN 1 ELSE 0 END                               AS treatment
  FROM first_only f
  LEFT JOIN next_campaign n ON f.household_key = n.household_key
)

SELECT 
  t.*, p.department, p.commodity_desc, p.sub_commodity_desc, 
p.manufacturer, p.brand, p.curr_size_of_product, ft.mailer,ft.display,dg.classification_1,
dg.classification_2, dg.classification_3,dg.HOMEOWNER_DESC,dg.classification_5,dg.classification_4,dg.KID_CATEGORY_DESC,
  -- Treatment
  c.treatment,
  c.description                                           AS campaign_type,
  c.clean_window_days,
  c.was_truncated,
  c.start_day                                             AS campaign_start,
  c.clean_end_day                                         AS campaign_end,

  -- Window flags
  CASE WHEN t.day BETWEEN c.start_day AND c.clean_end_day 
       THEN 1 ELSE 0 END                                  AS in_campaign_window,
  CASE WHEN t.day < c.start_day 
       THEN 1 ELSE 0 END                                  AS pre_campaign,

  -- Customer spend (actual price paid, not retailer revenue)
  CASE WHEN t.sales_value < ABS(t.coupon_disc) THEN 0 ELSE t.sales_value + t.coupon_disc END     AS customer_spend,

  -- Coupon usage flag
  CASE WHEN t.coupon_disc != 0 
       THEN 1 ELSE 0 END                                  AS coupon_used,
    
    CASE WHEN retail_disc != 0 
     THEN 1 ELSE 0 END                           AS loyalty_card_used

FROM bronze_transactions t 
JOIN clean c ON t.household_key = c.household_key
JOIN bronze_demographics dg on t.household_key = dg.household_key
JOIN bronze_products p on t.PRODUCT_ID = p.PRODUCT_ID
LEFT JOIN bronze_features ft on t.STORE_ID = ft.STORE_ID AND t.WEEK_NO = ft.WEEK_NO AND t.PRODUCT_ID = ft.PRODUCT_ID
)
GROUP BY in_campaign_window, pre_campaign

In [0]:
-- 4. customer_spend sanity check
SELECT 
  MIN(customer_spend)   AS min_spend,
  MAX(customer_spend)   AS max_spend,
  AVG(customer_spend)   AS avg_spend,
  SUM(CASE WHEN customer_spend < 0 THEN 1 ELSE 0 END) AS negative_spend_rows
FROM (
  WITH ordered_campaigns AS (
  SELECT 
    b.household_key, 
    b.campaign, 
    b.description, 
    d.start_day,
    d.end_day,
    ROW_NUMBER() OVER (
      PARTITION BY b.household_key 
      ORDER BY d.start_day
    ) AS rn 
  FROM bronze_campaigns b 
  JOIN bronze_campaign_desc d ON b.campaign = d.campaign
  WHERE household_key IN (
    SELECT household_key FROM bronze_demographics
  )
),

first_only AS (
  SELECT * FROM ordered_campaigns WHERE rn = 1
),

next_campaign AS (
    SELECT f.household_key, MIN(other_d.start_day) - 1 AS clean_end_day
    FROM first_only f   
    JOIN bronze_campaigns other ON other.household_key = f.household_key
    AND other.campaign    != f.campaign JOIN bronze_campaign_desc other_d
    ON other_d.campaign    = other.campaign
    AND other_d.start_day <= f.end_day
    AND other_d.end_day   >= f.start_day GROUP BY f.household_key),

clean AS (
      SELECT
    f.household_key,
    f.campaign,
    f.description,
    f.start_day,
    f.end_day                                             AS original_end_day,
    COALESCE(n.clean_end_day, f.end_day)                  AS clean_end_day,
    CASE WHEN n.clean_end_day IS NOT NULL 
         THEN 1 ELSE 0 END                               AS was_truncated,
    COALESCE(n.clean_end_day, f.end_day) 
      - f.start_day + 1                                  AS clean_window_days,
    CASE WHEN f.description = 'TypeA' 
         THEN 1 ELSE 0 END                               AS treatment
  FROM first_only f
  LEFT JOIN next_campaign n ON f.household_key = n.household_key
)

SELECT 
  t.*, p.department, p.commodity_desc, p.sub_commodity_desc, 
p.manufacturer, p.brand, p.curr_size_of_product, ft.mailer,ft.display,dg.classification_1,
dg.classification_2, dg.classification_3,dg.HOMEOWNER_DESC,dg.classification_5,dg.classification_4,dg.KID_CATEGORY_DESC,
  -- Treatment
  c.treatment,
  c.description                                           AS campaign_type,
  c.clean_window_days,
  c.was_truncated,
  c.start_day                                             AS campaign_start,
  c.clean_end_day                                         AS campaign_end,

  -- Window flags
  CASE WHEN t.day BETWEEN c.start_day AND c.clean_end_day 
       THEN 1 ELSE 0 END                                  AS in_campaign_window,
  CASE WHEN t.day < c.start_day 
       THEN 1 ELSE 0 END                                  AS pre_campaign,

  -- Customer spend (actual price paid, not retailer revenue)
  CASE WHEN t.sales_value < ABS(t.coupon_disc) THEN 0 ELSE t.sales_value + t.coupon_disc END     AS customer_spend,

  -- Coupon usage flag
  CASE WHEN t.coupon_disc != 0 
       THEN 1 ELSE 0 END                                  AS coupon_used,
    
    CASE WHEN retail_disc != 0 
     THEN 1 ELSE 0 END                           AS loyalty_card_used

FROM bronze_transactions t 
JOIN clean c ON t.household_key = c.household_key
JOIN bronze_demographics dg on t.household_key = dg.household_key
JOIN bronze_products p on t.PRODUCT_ID = p.PRODUCT_ID
LEFT JOIN bronze_features ft on t.STORE_ID = ft.STORE_ID AND t.WEEK_NO = ft.WEEK_NO AND t.PRODUCT_ID = ft.PRODUCT_ID
)

In [0]:
-- 4. customer_spend sanity check
SELECT 
*
FROM (
  WITH ordered_campaigns AS (
  SELECT 
    b.household_key, 
    b.campaign, 
    b.description, 
    d.start_day,
    d.end_day,
    ROW_NUMBER() OVER (
      PARTITION BY b.household_key 
      ORDER BY d.start_day
    ) AS rn 
  FROM bronze_campaigns b 
  JOIN bronze_campaign_desc d ON b.campaign = d.campaign
  WHERE household_key IN (
    SELECT household_key FROM bronze_demographics
  )
),

first_only AS (
  SELECT * FROM ordered_campaigns WHERE rn = 1
),

next_campaign AS (
    SELECT f.household_key, MIN(other_d.start_day) - 1 AS clean_end_day
    FROM first_only f   
    JOIN bronze_campaigns other ON other.household_key = f.household_key
    AND other.campaign    != f.campaign JOIN bronze_campaign_desc other_d
    ON other_d.campaign    = other.campaign
    AND other_d.start_day <= f.end_day
    AND other_d.end_day   >= f.start_day GROUP BY f.household_key),

clean AS (
      SELECT
    f.household_key,
    f.campaign,
    f.description,
    f.start_day,
    f.end_day                                             AS original_end_day,
    COALESCE(n.clean_end_day, f.end_day)                  AS clean_end_day,
    CASE WHEN n.clean_end_day IS NOT NULL 
         THEN 1 ELSE 0 END                               AS was_truncated,
    COALESCE(n.clean_end_day, f.end_day) 
      - f.start_day + 1                                  AS clean_window_days,
    CASE WHEN f.description = 'TypeA' 
         THEN 1 ELSE 0 END                               AS treatment
  FROM first_only f
  LEFT JOIN next_campaign n ON f.household_key = n.household_key
)

SELECT 
  t.*, p.department, p.commodity_desc, p.sub_commodity_desc, 
p.manufacturer, p.brand, p.curr_size_of_product, ft.mailer,ft.display,dg.classification_1,
dg.classification_2, dg.classification_3,dg.HOMEOWNER_DESC,dg.classification_5,dg.classification_4,dg.KID_CATEGORY_DESC,
  -- Treatment
  c.treatment,
  c.description                                           AS campaign_type,
  c.clean_window_days,
  c.was_truncated,
  c.start_day                                             AS campaign_start,
  c.clean_end_day                                         AS campaign_end,

  -- Window flags
  CASE WHEN t.day BETWEEN c.start_day AND c.clean_end_day 
       THEN 1 ELSE 0 END                                  AS in_campaign_window,
  CASE WHEN t.day < c.start_day 
       THEN 1 ELSE 0 END                                  AS pre_campaign,

  -- Customer spend (actual price paid, not retailer revenue)
  CASE WHEN t.sales_value < ABS(t.coupon_disc) THEN 0 ELSE t.sales_value + t.coupon_disc END     AS customer_spend,

  -- Coupon usage flag
  CASE WHEN t.coupon_disc != 0 
       THEN 1 ELSE 0 END                                  AS coupon_used,
    
    CASE WHEN retail_disc != 0 
     THEN 1 ELSE 0 END                           AS loyalty_card_used

FROM bronze_transactions t 
JOIN clean c ON t.household_key = c.household_key
JOIN bronze_demographics dg on t.household_key = dg.household_key
JOIN bronze_products p on t.PRODUCT_ID = p.PRODUCT_ID
LEFT JOIN bronze_features ft on t.STORE_ID = ft.STORE_ID AND t.WEEK_NO = ft.WEEK_NO AND t.PRODUCT_ID = ft.PRODUCT_ID
)
WHERE customer_spend = 0 LIMIT 10